In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
data = [
    ("Alice went to Paris".split(), ["B-PER", "O", "O", "B-LOC"]),
    ("Bob is working at Microsoft".split(), ["B-PER", "O", "O", "O", "B-ORG"]),
    ("Charlie lives in London".split(), ["B-PER", "O", "O", "B-LOC"]),
]

In [3]:
word_to_ix = {}
tag_to_ix = {"O": 0, "B-PER": 1, "I-PER": 2, "B-LOC": 3, "I-LOC": 4, "B-ORG": 5}

for sentence, tags in data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

In [4]:
def prepare_sequence(seq, to_ix):
    return torch.tensor([to_ix[w] for w in seq], dtype=torch.long)

In [5]:
class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=64, hidden_dim=64):
        super(LSTMTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.embedding(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.fc(lstm_out.view(len(sentence), -1))
        tag_scores = torch.log_softmax(tag_space, dim=1)
        return tag_scores

In [6]:
model = LSTMTagger(len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [7]:
for epoch in range(100):
    total_loss = 0
    for sentence, tags in data:
        model.zero_grad()

        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        tag_scores = model(sentence_in)
        loss = loss_function(tag_scores, targets)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 5.3283
Epoch 10, Loss: 2.9480
Epoch 20, Loss: 1.3722
Epoch 30, Loss: 0.6063
Epoch 40, Loss: 0.3222
Epoch 50, Loss: 0.2029
Epoch 60, Loss: 0.1427
Epoch 70, Loss: 0.1079
Epoch 80, Loss: 0.0857
Epoch 90, Loss: 0.0704


In [9]:
with torch.no_grad():
    test_sentence = "Bob is working at Microsoft".split()   # from your dataset
    inputs = prepare_sequence(test_sentence, word_to_ix)

    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)

    ix_to_tag = {v: k for k, v in tag_to_ix.items()}
    predicted_tags = [ix_to_tag[ix.item()] for ix in predicted]

    print("\nSentence:", test_sentence)
    print("Predicted tags:", predicted_tags)


Sentence: ['Bob', 'is', 'working', 'at', 'Microsoft']
Predicted tags: ['B-PER', 'O', 'O', 'O', 'B-ORG']
